# ChatGPT Conversation History - Topic Analysis

This notebook performs advanced text analysis to discover topics and themes in your conversations.

**Techniques Used:**
- TF-IDF keyword extraction
- Word clouds and n-grams
- Topic modeling (Latent Dirichlet Allocation)
- Conversation clustering

---

## Setup

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# NLP libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("Libraries loaded!")

In [ ]:
# Load data
import os
if not os.path.exists('conversations.json'):
    raise FileNotFoundError("conversations.json not found — put it in the repo root or update the path.")
with open('conversations.json', 'r', encoding='utf-8') as f:
    conversations = json.load(f)

df = pd.read_csv('../chatgpt_conversations_dataframe.csv')

print(f"Loaded {len(conversations):,} conversations")

## 1. Text Preparation

In [ ]:
# Extract all text from conversations
def extract_conversation_text(conv, include_roles=['user', 'assistant']):
    """Extract text from a conversation, optionally filtering by role."""
    texts = []
    for msg in conv['messages']:
        if msg['role'] in include_roles and msg['content']:
            # Clean the text
            text = msg['content']
            # Remove code blocks
            text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)
            # Remove URLs
            text = re.sub(r'http\S+|www\S+', '', text)
            # Remove special characters but keep spaces
            text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
            texts.append(text)
    return ' '.join(texts)

# Create corpus from conversation titles and content
titles = [conv['title'] for conv in conversations]
full_texts = [extract_conversation_text(conv) for conv in conversations]

# Combine title and text for better topic extraction
combined_texts = [f"{title} {text}" for title, text in zip(titles, full_texts)]

print(f"Prepared {len(combined_texts):,} documents for analysis")
print(f"Average document length: {np.mean([len(t) for t in combined_texts]):,.0f} characters")

## 2. Word Frequency Analysis

In [ ]:
# Extract words from all text
stop_words = set(stopwords.words('english'))
# Add custom stop words
custom_stops = {'would', 'could', 'also', 'use', 'like', 'one', 'get', 'make', 
                'know', 'think', 'see', 'want', 'need', 'way', 'time', 'thing'}
stop_words.update(custom_stops)

all_words = []
for text in combined_texts:
    words = word_tokenize(text.lower())
    words = [w for w in words if w.isalpha() and len(w) > 3 and w not in stop_words]
    all_words.extend(words)

word_freq = Counter(all_words)

print("TOP 50 MOST COMMON WORDS")
print("=" * 60)
for i, (word, count) in enumerate(word_freq.most_common(50), 1):
    if i % 5 == 1:
        print()
    print(f"{word:<15} {count:>6}", end="  ")
print("\n" + "=" * 60)

In [ ]:
# Visualize top words
top_words = word_freq.most_common(30)
words, counts = zip(*top_words)

plt.figure(figsize=(12, 8))
bars = plt.barh(range(len(words)), counts, edgecolor='black')

# Color gradient
colors = plt.cm.plasma(np.linspace(0.3, 0.9, len(bars)))
for bar, color in zip(bars, colors):
    bar.set_color(color)

plt.yticks(range(len(words)), words)
plt.xlabel('Frequency')
plt.title('Top 30 Most Frequent Words', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 3. Word Clouds

In [ ]:
# Generate word cloud
all_text = ' '.join(combined_texts)

wordcloud = WordCloud(width=1600, height=800, 
                     background_color='white',
                     stopwords=stop_words,
                     min_font_size=10,
                     max_words=200,
                     colormap='viridis',
                     relative_scaling=0.5).generate(all_text)

plt.figure(figsize=(16, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of All Conversations', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Word cloud from titles only
titles_text = ' '.join(titles)

wordcloud_titles = WordCloud(width=1600, height=800, 
                             background_color='white',
                             stopwords=stop_words,
                             min_font_size=10,
                             max_words=100,
                             colormap='plasma').generate(titles_text)

plt.figure(figsize=(16, 8))
plt.imshow(wordcloud_titles, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud from Conversation Titles', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 4. Bigrams and Trigrams

In [ ]:
# Extract bigrams (2-word phrases)
from nltk import bigrams, trigrams

all_bigrams = []
all_trigrams = []

for text in combined_texts:
    words = word_tokenize(text.lower())
    words = [w for w in words if w.isalpha() and len(w) > 3 and w not in stop_words]
    
    all_bigrams.extend([' '.join(bg) for bg in bigrams(words)])
    all_trigrams.extend([' '.join(tg) for tg in trigrams(words)])

bigram_freq = Counter(all_bigrams)
trigram_freq = Counter(all_trigrams)

print("TOP 30 BIGRAMS (2-word phrases)")
print("=" * 70)
for i, (phrase, count) in enumerate(bigram_freq.most_common(30), 1):
    print(f"{i:>2}. {phrase:<35} {count:>5}")

print("\nTOP 30 TRIGRAMS (3-word phrases)")
print("=" * 70)
for i, (phrase, count) in enumerate(trigram_freq.most_common(30), 1):
    print(f"{i:>2}. {phrase:<45} {count:>5}")

In [ ]:
# Visualize top bigrams and trigrams
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Bigrams
top_bigrams = bigram_freq.most_common(20)
phrases, counts = zip(*top_bigrams)
axes[0].barh(range(len(phrases)), counts, edgecolor='black', color='skyblue')
axes[0].set_yticks(range(len(phrases)))
axes[0].set_yticklabels(phrases)
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 20 Bigrams', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3, axis='x')

# Trigrams
top_trigrams = trigram_freq.most_common(20)
phrases, counts = zip(*top_trigrams)
axes[1].barh(range(len(phrases)), counts, edgecolor='black', color='lightcoral')
axes[1].set_yticks(range(len(phrases)))
axes[1].set_yticklabels(phrases)
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top 20 Trigrams', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 5. TF-IDF Analysis

In [ ]:
# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    min_df=2,
    max_df=0.8,
    stop_words=list(stop_words),
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(combined_texts)
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"Number of features: {len(feature_names)}")

In [ ]:
# Find most important terms across all documents
tfidf_scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_indices = tfidf_scores.argsort()[-50:][::-1]
top_terms = [(feature_names[i], tfidf_scores[i]) for i in top_indices]

print("TOP 50 TERMS BY TF-IDF SCORE")
print("=" * 70)
for i, (term, score) in enumerate(top_terms, 1):
    print(f"{i:>2}. {term:<30} {score:.6f}")

In [ ]:
# Visualize top TF-IDF terms
top_20 = top_terms[:20]
terms, scores = zip(*top_20)

plt.figure(figsize=(12, 8))
bars = plt.barh(range(len(terms)), scores, edgecolor='black')

colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(bars)))
for bar, color in zip(bars, colors):
    bar.set_color(color)

plt.yticks(range(len(terms)), terms)
plt.xlabel('TF-IDF Score')
plt.title('Top 20 Terms by TF-IDF', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 6. Topic Modeling with LDA

In [ ]:
# Prepare data for LDA (uses raw counts, not TF-IDF)
count_vectorizer = CountVectorizer(
    max_features=1000,
    min_df=3,
    max_df=0.8,
    stop_words=list(stop_words)
)

count_matrix = count_vectorizer.fit_transform(combined_texts)
count_features = count_vectorizer.get_feature_names_out()

print(f"Count Matrix shape: {count_matrix.shape}")

In [ ]:
# Fit LDA model
n_topics = 10
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=20,
    learning_method='online',
    n_jobs=-1
)

print(f"Fitting LDA model with {n_topics} topics...")
lda_output = lda.fit_transform(count_matrix)
print("✓ LDA model fitted")

In [ ]:
# Display topics
def display_topics(model, feature_names, n_top_words=10):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        topics.append(top_words)
    return topics

topics = display_topics(lda, count_features, n_top_words=15)

print("\n" + "=" * 80)
print(f"DISCOVERED {n_topics} TOPICS")
print("=" * 80)
for i, topic_words in enumerate(topics, 1):
    print(f"\nTopic {i}:")
    print(f"  {', '.join(topic_words)}")

In [ ]:
# Assign dominant topic to each conversation
dominant_topics = lda_output.argmax(axis=1)
topic_confidence = lda_output.max(axis=1)

df['dominant_topic'] = dominant_topics
df['topic_confidence'] = topic_confidence

# Count conversations per topic
topic_counts = pd.Series(dominant_topics).value_counts().sort_index()

print("\nCONVERSATIONS PER TOPIC")
print("=" * 60)
for topic_num, count in topic_counts.items():
    pct = (count / len(df)) * 100
    print(f"Topic {topic_num + 1}: {count:>4} conversations ({pct:>5.1f}%)")
    print(f"  Keywords: {', '.join(topics[topic_num][:8])}")
    print()

In [ ]:
# Visualize topic distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Bar chart
axes[0].bar(range(1, n_topics + 1), topic_counts.values, edgecolor='black')
axes[0].set_xlabel('Topic Number')
axes[0].set_ylabel('Number of Conversations')
axes[0].set_title('Conversations per Topic', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Pie chart
axes[1].pie(topic_counts.values, labels=[f'Topic {i+1}' for i in range(n_topics)],
           autopct='%1.1f%%', startangle=90)
axes[1].set_title('Topic Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Show example conversations for each topic
print("\nEXAMPLE CONVERSATIONS FOR EACH TOPIC")
print("=" * 80)
for topic_num in range(n_topics):
    topic_convs = df[df['dominant_topic'] == topic_num].nlargest(3, 'topic_confidence')
    print(f"\nTopic {topic_num + 1}: {', '.join(topics[topic_num][:5])}")
    for idx, row in topic_convs.iterrows():
        title = row['title'][:70] + '...' if len(row['title']) > 70 else row['title']
        print(f"  • {title} (confidence: {row['topic_confidence']:.3f})")

## 7. Conversation Clustering

In [ ]:
# K-Means clustering on TF-IDF vectors
n_clusters = 8
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(tfidf_matrix)

df['cluster'] = clusters

# Count per cluster
cluster_counts = pd.Series(clusters).value_counts().sort_index()

print(f"K-Means Clustering with {n_clusters} clusters")
print("=" * 60)
for cluster_num, count in cluster_counts.items():
    pct = (count / len(df)) * 100
    print(f"Cluster {cluster_num}: {count:>4} conversations ({pct:>5.1f}%)")

In [ ]:
# Extract top terms for each cluster
print("\nTOP TERMS FOR EACH CLUSTER")
print("=" * 80)
for cluster_num in range(n_clusters):
    cluster_center = kmeans.cluster_centers_[cluster_num]
    top_indices = cluster_center.argsort()[-10:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    
    print(f"\nCluster {cluster_num}: {cluster_counts[cluster_num]} conversations")
    print(f"  Keywords: {', '.join(top_terms)}")
    
    # Show example conversations
    cluster_convs = df[df['cluster'] == cluster_num].head(3)
    for idx, row in cluster_convs.iterrows():
        title = row['title'][:60] + '...' if len(row['title']) > 60 else row['title']
        print(f"    • {title}")

## 8. Visualization: t-SNE Projection

In [ ]:
# Use t-SNE to reduce dimensionality for visualization
# Note: This can take a few minutes for large datasets
print("Computing t-SNE projection (this may take a minute)...")

# Sample if dataset is too large
sample_size = min(1000, len(combined_texts))
sample_indices = np.random.choice(len(combined_texts), sample_size, replace=False)

tfidf_sample = tfidf_matrix[sample_indices]
clusters_sample = clusters[sample_indices]
topics_sample = dominant_topics[sample_indices]

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_results = tsne.fit_transform(tfidf_sample.toarray())

print("✓ t-SNE completed")

In [ ]:
# Visualize clusters in 2D
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# By K-Means cluster
scatter1 = axes[0].scatter(tsne_results[:, 0], tsne_results[:, 1], 
                          c=clusters_sample, cmap='tab10', alpha=0.6, s=50)
axes[0].set_title('Conversations Clustered by K-Means', fontsize=12, fontweight='bold')
axes[0].set_xlabel('t-SNE Dimension 1')
axes[0].set_ylabel('t-SNE Dimension 2')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# By LDA topic
scatter2 = axes[1].scatter(tsne_results[:, 0], tsne_results[:, 1], 
                          c=topics_sample, cmap='tab10', alpha=0.6, s=50)
axes[1].set_title('Conversations by LDA Topic', fontsize=12, fontweight='bold')
axes[1].set_xlabel('t-SNE Dimension 1')
axes[1].set_ylabel('t-SNE Dimension 2')
plt.colorbar(scatter2, ax=axes[1], label='Topic')

plt.tight_layout()
plt.show()

## 9. Save Results

In [ ]:
# Save enhanced dataframe with topic and cluster information
df.to_csv('../chatgpt_conversations_with_topics.csv', index=False)
print("✓ Saved enhanced DataFrame to 'chatgpt_conversations_with_topics.csv'")

# Save topic information
topic_info = pd.DataFrame({
    'Topic': range(1, n_topics + 1),
    'Keywords': [', '.join(t[:10]) for t in topics],
    'Count': topic_counts.values,
    'Percentage': (topic_counts.values / len(df) * 100).round(1)
})
topic_info.to_csv('../chatgpt_topics.csv', index=False)
print("✓ Saved topic information to 'chatgpt_topics.csv'")

## 10. Topic Insights Summary

In [ ]:
print("\n" + "=" * 80)
print("TOPIC ANALYSIS INSIGHTS")
print("=" * 80)

print("\n🔍 MOST COMMON THEMES:")
for i, (word, count) in enumerate(word_freq.most_common(10), 1):
    print(f"  {i}. {word} ({count:,} occurrences)")

print("\n📝 TOP PHRASES:")
for i, (phrase, count) in enumerate(bigram_freq.most_common(5), 1):
    print(f"  {i}. '{phrase}' ({count} times)")

print(f"\n🎯 DISCOVERED TOPICS:")
print(f"  • {n_topics} main topics identified using LDA")
print(f"  • Most common topic: Topic {topic_counts.idxmax() + 1} ({topic_counts.max()} conversations)")
print(f"  • Keywords: {', '.join(topics[topic_counts.idxmax()][:8])}")

print(f"\n📊 CLUSTERING:")
print(f"  • Conversations grouped into {n_clusters} clusters")
print(f"  • Largest cluster: Cluster {cluster_counts.idxmax()} ({cluster_counts.max()} conversations)")

print("\n" + "=" * 80)
print("Next: Run notebook 04 for conversation pattern analysis")
print("=" * 80)